# पाठ 08 - मल्टी-एजेंट डिज़ाइन पैटर्न


## सेटअप


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## क्यों मल्टी-एजेंट सिस्टम?

वास्तविक दुनिया के कार्य जैसे यात्रा योजना में कई विभिन्न प्रकार की विशेषज्ञता शामिल होती है — लॉजिस्टिक्स, स्थानीय ज्ञान, बजटिंग, और बहुत कुछ। एक ही एजेंट जो सबकुछ संभालने की कोशिश करता है वह जल्दी ही अप्रबंधनीय हो जाता है।

मल्टी-एजेंट सिस्टम इसे **विशेषीकरण** के माध्यम से हल करते हैं: प्रत्येक एजेंट एक क्षेत्र की विशेषज्ञता पर केंद्रित होता है, जो एक सामान्य विशेषज्ञ की तुलना में अधिक उच्च गुणवत्ता वाले परिणाम उत्पन्न करता है। वे **स्केलेबिलिटी** में भी सुधार करते हैं — आप नए एजेंट जोड़ सकते हैं (जैसे, एक उड़ान विशेषज्ञ, एक रेस्टोरेंट समालोचक) बिना मौजूदा वर्कफ़्लो को फिर से लिखे। एजेंट एक संरचित पाइपलाइन के माध्यम से मिलकर काम करते हैं, एक से दूसरे को संदर्भ पास करते हैं।


## विशेष एजेंट बनाना


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## एक अनुक्रमिक कार्यप्रवाह बनाना

`WorkflowBuilder` आपको एजेंटों को एक निर्देशित ग्राफ में जोड़ने देता है। यहाँ हम एक साधारण दो-चरण पाइपलाइन बनाते हैं: **TravelPlanner** यात्रा कार्यक्रम तैयार करता है, फिर **TravelConcierge** इसकी समीक्षा करता है और इसे बेहतर बनाता है।


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## वर्कफ़्लो में अधिक एजेंट जोड़ना

मल्टी-एजेंट पैटर्न का एक सबसे बड़ा फायदा यह है कि इसे बढ़ाना कितना आसान है। नीचे हम एक **BudgetReviewer** एजेंट जोड़ते हैं जो योजना की यात्री के बजट के खिलाफ जांच करता है, उन आइटमों को चिह्नित करता है जो खर्च को सीमा से ऊपर ले जा सकते हैं, और पैसे बचाने वाले विकल्प सुझाता है। वर्कफ़्लो अब तीन एजेंटों को अनुक्रम में चलाता है:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## सारांश

इस पाठ में आपने सीखा कि कैसे:

1. **विशेषीकृत एजेंट बनाएं** — प्रत्येक का एक केंद्रित भूमिका होती है (योजना बनाना, कंसीयर्ज, बजट समीक्षा)।
2. **एजेंट्स को एक क्रमिक कार्यप्रवाह में जोड़ें** `WorkflowBuilder` और `add_edge` का उपयोग करके।
3. **आउटपुट को स्ट्रीम करें** एक मल्टी-एजेंट पाइपलाइन से, यह ट्रैक करते हुए कि कौन सा एजेंट बोल रहा है।
4. **कार्यप्रवाह का विस्तार करें** नए एजेंट जोड़कर श्रृंखला में बिना मौजूदा एजेंटों को संशोधित किए।

मल्टी-एजेंट डिज़ाइन पैटर्न प्रत्येक एजेंट को सरल रखता है जबकि अधिक समृद्ध, अधिक सावधानीपूर्वक समीक्षा किए गए परिणाम उत्पन्न करता है जो कोई एकल एजेंट अकेले प्राप्त नहीं कर सकता।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:  
यह दस्तावेज़ एआई अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) का उपयोग करके अनुवादित किया गया है। हम सटीकता के लिए प्रयासरत हैं, लेकिन कृपया ध्यान दें कि स्वचालित अनुवादों में त्रुटियाँ या गलतियाँ हो सकती हैं। मूल दस्तावेज़ अपनी मूल भाषा में प्रामाणिक स्रोत माना जाना चाहिए। महत्वपूर्ण जानकारी के लिए, पेशेवर मानव अनुवाद की सलाह दी जाती है। इस अनुवाद के उपयोग से उत्पन्न किसी भी गलतफहमी या गलत व्याख्या के लिए हम जिम्मेदार नहीं हैं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
